# 04 - Feature Engineering & Statistical Analysis

Create engineered features (price per sqft, property age, amenity counts, location features), perform statistical analysis including central tendency, dispersion, correlations, VIF for multicollinearity, and normality tests.

In [ ]:
# Setup
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.preprocessing import load_processed_data
from src.data.feature_engineering import (
    calculate_price_per_sqft,
    calculate_property_age,
    create_renovation_flag,
    create_age_bands,
    create_amenity_count,
    create_basement_flag,
    create_living_ratio,
    create_zipcode_frequency,
    calculate_distance_to_seattle,
    create_all_features,
    statistical_analysis,
    full_feature_engineering_pipeline,
    save_engineered_data
)
from src.config import DATA_PROCESSED, AGE_BANDS, CONDITION_THRESHOLD, GRADE_THRESHOLD, SEATTLE_CENTER

In [ ]:
# Load processed training data
X_train, X_test, y_train, y_test, scaler = load_processed_data()

# Combine for feature engineering
train_df = X_train.copy()
train_df['price'] = y_train.values

print(f"Training data shape: {train_df.shape}")
print(f"Columns: {list(train_df.columns)}")

In [ ]:
# 1. Price per SqFt
print("=== 1. Price per SqFt ===")
train_df = calculate_price_per_sqft(train_df)
print(f"Price per sqft stats:\n{train_df['price_per_sqft'].describe()}")

In [ ]:
# 2. Property Age
print("=== 2. Property Age ===")
train_df = calculate_property_age(train_df, reference_year=2015)
print(f"Age stats:\n{train_df['age'].describe()}")

In [ ]:
# 3. Renovation Flag
print("=== 3. Renovation Flag ===")
train_df = create_renovation_flag(train_df)
print(f"Renovated: {train_df['is_renovated'].sum()} ({train_df['is_renovated'].mean()*100:.1f}%)")

In [ ]:
# 4. Age Bands
print("=== 4. Age Bands ===")
train_df = create_age_bands(train_df, AGE_BANDS)
print(train_df['age_band'].value_counts().sort_index())

In [ ]:
# 5. Amenity Count
print("=== 5. Amenity Count ===")
train_df = create_amenity_count(train_df)
print(train_df['amenity_count'].value_counts().sort_index())

In [ ]:
# 6. Basement Flag
print("=== 6. Basement Flag ===")
train_df = create_basement_flag(train_df)
print(f"Has basement: {train_df['has_basement'].sum()} ({train_df['has_basement'].mean()*100:.1f}%)")

In [ ]:
# 7. Living Ratio
print("=== 7. Living Ratio ===")
train_df = create_living_ratio(train_df)
print(f"Living ratio stats:\n{train_df['living_ratio'].describe()}")

In [ ]:
# 8. Zipcode Frequency Encoding
print("=== 8. Zipcode Frequency ===")
train_df = create_zipcode_frequency(train_df)
print(f"Zipcode freq stats:\n{train_df['zipcode_freq'].describe()}")

In [ ]:
# 9. Distance to Seattle Center
print("=== 9. Distance to Seattle Center ===")
train_df = calculate_distance_to_seattle(train_df, SEATTLE_CENTER)
print(f"Distance stats:\n{train_df['dist_to_seattle_center'].describe()}")

In [ ]:
# 10. Create All Features (Alternative)
print("=== 10. Full Feature Engineering Pipeline ===")
# Reload to show pipeline
X_train, X_test, y_train, y_test, scaler = load_processed_data()
train_df2 = X_train.copy()
train_df2['price'] = y_train.values

engineered_df, stats_results = full_feature_engineering_pipeline(train_df2)

print(f"\nEngineered shape: {engineered_df.shape}")
print(f"New features: {[c for c in engineered_df.columns if c not in train_df2.columns]}")

In [ ]:
# 11. Statistical Analysis Results
print("=== 11. Statistical Analysis Results ===")

print("\n--- Target Statistics ---")
target_stats = stats_results['target_stats']
for k, v in target_stats.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# Correlations with Price
print("\n--- Top Correlations with Price ---")
corrs = stats_results['correlations']
corr_series = pd.Series(corrs).sort_values(key=abs, ascending=False)
print(corr_series.head(15).to_string())

In [ ]:
# VIF Analysis
print("\n--- VIF Analysis ---")
vif = stats_results.get('vif', {})
if vif:
    vif_df = pd.DataFrame(list(vif.items()), columns=['feature', 'VIF']).sort_values('VIF', ascending=False)
    print(vif_df.to_string(index=False))
    
    # Flag high VIF
    high_vif = vif_df[vif_df['VIF'] > 5]
    if len(high_vif) > 0:
        print(f"\n⚠ High VIF (>5): {high_vif['feature'].tolist()}")
    very_high = vif_df[vif_df['VIF'] > 10]
    if len(very_high) > 0:
        print(f"⚠ Very High VIF (>10): {very_high['feature'].tolist()}")

In [ ]:
# Normality Tests
print("\n--- Normality Tests ---")
norm = stats_results['normality_tests']
for k, v in norm.items():
    print(f"  {k}: {v}")

In [ ]:
# Price per SqFt by Categories
print("\n--- Price per SqFt by Zipcode (Top 10) ---")
ppsqft_zip = stats_results['price_per_sqft_by_zipcode']
if ppsqft_zip:
    zip_df = pd.DataFrame(ppsqft_zip).T.sort_values('mean', ascending=False).head(10)
    print(zip_df.round(0).to_string())

In [ ]:
print("\n--- Price per SqFt by Age Band ---")
ppsqft_age = stats_results['price_per_sqft_by_age_band']
if ppsqft_age:
    age_df = pd.DataFrame(ppsqft_age).T
    print(age_df.round(0).to_string())

In [ ]:
print("\n--- Price per SqFt by Condition ---")
ppsqft_cond = stats_results['price_per_sqft_by_condition']
if ppsqft_cond:
    cond_df = pd.DataFrame(ppsqft_cond).T.sort_index()
    print(cond_df.round(0).to_string())

In [ ]:
print("\n--- Price per SqFt by Grade ---")
ppsqft_grade = stats_results['price_per_sqft_by_grade']
if ppsqft_grade:
    grade_df = pd.DataFrame(ppsqft_grade).T.sort_index()
    print(grade_df.round(0).to_string())

In [ ]:
# 12. Visualize Engineered Features
print("=== 12. Visualizing Engineered Features ===")

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

features_to_plot = ['price_per_sqft', 'age', 'is_renovated', 'amenity_count',
                    'has_basement', 'living_ratio', 'zipcode_freq', 'dist_to_seattle_center']

for idx, feat in enumerate(features_to_plot):
    if feat in engineered_df.columns:
        ax = axes[idx]
        if engineered_df[feat].nunique() > 10:
            ax.scatter(engineered_df[feat], engineered_df['price'], alpha=0.3, s=10)
        else:
            sns.boxplot(data=engineered_df, x=feat, y='price', ax=ax)
        ax.set_title(f'Price vs {feat}')
        ax.set_xlabel(feat)

plt.tight_layout()
plt.show()

In [ ]:
# 13. Correlation Heatmap with Engineered Features
print("=== 13. Correlation Heatmap (Engineered) ===")

numeric_cols = engineered_df.select_dtypes(include=[np.number]).columns
corr = engineered_df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, square=True,
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Correlation Heatmap with Engineered Features')
plt.tight_layout()
plt.show()

In [ ]:
# 14. Save Engineered Data for Modeling
print("=== 14. Saving Engineered Data ===")
save_engineered_data(engineered_df)

# Also save test set with engineered features
test_df = X_test.copy()
test_df['price'] = y_test.values
test_engineered, _ = full_feature_engineering_pipeline(test_df)
test_engineered.to_csv(DATA_PROCESSED / 'kc_house_test_engineered.csv', index=False)
print("Test engineered data saved!")